# Modelo de Classificação de Prioridade das Ocorrências

## 1. Objetivo

Este notebook tem como objetivo desenvolver um modelo de Machine Learning capaz de classificar o nível de prioridade de ocorrências relacionadas à acessibilidade urbana.

O modelo utiliza três informações como entrada:

- Categoria da barreira.
- Nível de impacto.
- Existência de uma rota alternativa.

Com base nessas informações, o modelo prevê uma das três classificações de prioridade:

- BAIXA
- MEDIA
- ALTA

Essa classificação pode auxiliar na identificação de ocorrências que necessitam de maior atenção na plataforma AcessoBairro.

In [1]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

df = pd.read_csv("../datasets/dataset_prioridades.csv")

display(df.head())

print("Dimensões do dataset:", df.shape)

print("\nInformações do dataset:")
df.info()

print("\nValores ausentes:")
print(df.isnull().sum())

,categoria,impacto,rota_alternativa,prioridade
0,RAMPA_AUSENTE,MEDIO,NAO,ALTA
1,RAMPA_INADEQUADA,ALTO,SIM,ALTA
2,CALCADA_DANIFICADA,MEDIO,SIM,MEDIA
3,RAMPA_AUSENTE,MEDIO,SIM,MEDIA
4,CALCADA_DANIFICADA,MEDIO,SIM,MEDIA


Dimensões do dataset: (500, 4)

Informações do dataset:
<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   categoria         500 non-null    str  
 1   impacto           500 non-null    str  
 2   rota_alternativa  500 non-null    str  
 3   prioridade        500 non-null    str  
dtypes: str(4)
memory usage: 15.8 KB

Valores ausentes:
categoria           0
impacto             0
rota_alternativa    0
prioridade          0
dtype: int64


## 2. Análise Exploratória dos Dados

Nesta etapa, são analisadas as informações presentes no dataset `dataset_prioridades.csv`.

O arquivo contém 500 registros e quatro colunas:

- `categoria`: identifica o tipo de barreira.
- `impacto`: representa o nível de impacto da ocorrência.
- `rota_alternativa`: indica se existe uma rota alternativa.
- `prioridade`: representa a classificação de prioridade.

Inicialmente, são verificadas a distribuição dos registros, a estrutura do dataset e a existência de valores ausentes.

In [2]:
print("Distribuição das categorias:")
print(df["categoria"].value_counts())

print("\nDistribuição dos impactos:")
print(df["impacto"].value_counts())

print("\nDistribuição das rotas alternativas:")
print(df["rota_alternativa"].value_counts())

print("\nDistribuição das prioridades:")
print(df["prioridade"].value_counts())

Distribuição das categorias:
categoria
RAMPA_AUSENTE           100
RAMPA_INADEQUADA        100
CALCADA_DANIFICADA      100
PASSAGEM_OBSTRUIDA      100
TRAVESSIA_INADEQUADA    100
Name: count, dtype: int64

Distribuição dos impactos:
impacto
MEDIO    200
BAIXO    200
ALTO     100
Name: count, dtype: int64

Distribuição das rotas alternativas:
rota_alternativa
NAO    250
SIM    250
Name: count, dtype: int64

Distribuição das prioridades:
prioridade
ALTA     200
MEDIA    200
BAIXA    100
Name: count, dtype: int64


### 2.1 Distribuição e Relação entre as Variáveis

Nesta etapa, são apresentadas as distribuições das categorias, dos níveis de impacto, das rotas alternativas e das prioridades.

Também é realizado um cruzamento entre o impacto, a existência de uma rota alternativa e a prioridade atribuída.

Essa análise permite observar como as informações presentes no dataset estão relacionadas à classificação de prioridade.

In [3]:
pd.crosstab(
    [df["impacto"], df["rota_alternativa"]],
    df["prioridade"]
)

prioridade                ALTA  BAIXA  MEDIA
impacto rota_alternativa                    
ALTO    NAO                 50      0      0
        SIM                 50      0      0
BAIXO   NAO                  0      0    100
        SIM                  0    100      0
MEDIO   NAO                100      0      0
        SIM                  0      0    100

## 3. Preparação dos Dados

Nesta etapa, são definidas as variáveis utilizadas pelo modelo:

- **Variáveis de entrada (X):** categoria, impacto e existência de rota alternativa.
- **Variável alvo (y):** prioridade da ocorrência.

O dataset é dividido em dois conjuntos:

- 80% dos registros para treinamento.
- 20% dos registros para teste.

A divisão utiliza estratificação para manter a proporção das classes de prioridade nos conjuntos de treinamento e teste.

Também é configurado o OneHotEncoder, responsável por transformar as variáveis categóricas em representações numéricas que podem ser utilizadas pelo algoritmo.

In [4]:
X = df[["categoria", "impacto", "rota_alternativa"]]
y = df["prioridade"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Registros para treinamento:", len(X_treino))
print("Registros para teste:", len(X_teste))

preprocessador = ColumnTransformer(
    transformers=[
        ("categoricas", OneHotEncoder(handle_unknown="ignore"),
         ["categoria", "impacto", "rota_alternativa"])
    ]
)

Registros para treinamento: 400
Registros para teste: 100


## 4. Construção e Treinamento do Modelo

O modelo utiliza um Pipeline para reunir o pré-processamento dos dados e o algoritmo de classificação.

São utilizadas as seguintes ferramentas:

- **ColumnTransformer:** organiza a transformação das variáveis de entrada.
- **OneHotEncoder:** transforma as informações categóricas em valores numéricos.
- **DecisionTreeClassifier:** algoritmo de Árvore de Decisão responsável por classificar a prioridade das ocorrências.

A árvore de decisão utiliza profundidade máxima igual a 4, limitando sua complexidade.

Após a configuração, o modelo é treinado utilizando o conjunto de dados destinado ao treinamento.

In [5]:
modelo_prioridade = Pipeline(steps=[
    ("preprocessamento", preprocessador),
    ("classificador", DecisionTreeClassifier(
        max_depth=4,
        random_state=42
    ))
])

modelo_prioridade.fit(X_treino, y_treino)

print("Modelo treinado com sucesso!")

Modelo treinado com sucesso!


## 5. Avaliação do Modelo

Após o treinamento, o modelo realiza previsões utilizando o conjunto de teste.

O desempenho é avaliado por meio das seguintes métricas:

- **Acurácia:** proporção de classificações corretas.
- **Precisão:** proporção de previsões corretas em cada classe.
- **Recall:** capacidade de identificar corretamente os registros de cada classe.
- **F1-score:** equilíbrio entre precisão e recall.

Também é apresentada uma matriz de confusão para comparar as prioridades reais com as previstas pelo modelo.

In [6]:
y_pred = modelo_prioridade.predict(X_teste)

acuracia = accuracy_score(y_teste, y_pred)

print("Acurácia:", acuracia)

print("\nRelatório de classificação:")
print(classification_report(y_teste, y_pred, zero_division=0))

print("\nMatriz de confusão:")
print(confusion_matrix(
    y_teste,
    y_pred,
    labels=["BAIXA", "MEDIA", "ALTA"]
))

Acurácia: 1.0

Relatório de classificação:
              precision    recall  f1-score   support

        ALTA       1.00      1.00      1.00        40
       BAIXA       1.00      1.00      1.00        20
       MEDIA       1.00      1.00      1.00        40

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100


Matriz de confusão:
[[20  0  0]
 [ 0 40  0]
 [ 0  0 40]]


## 6. Testes com Novas Ocorrências

Nesta etapa, são criados exemplos de novas ocorrências com diferentes combinações de categoria, impacto e existência de rota alternativa.

Esses registros são enviados ao modelo treinado para que sejam realizadas previsões de prioridade.

O objetivo é demonstrar como o classificador pode ser utilizado para analisar novas ocorrências que não fazem parte do conjunto de treinamento.

In [7]:
novas_ocorrencias = pd.DataFrame([
    {
        "categoria": "CALCADA_DANIFICADA",
        "impacto": "BAIXO",
        "rota_alternativa": "SIM"
    },
    {
        "categoria": "RAMPA_AUSENTE",
        "impacto": "MEDIO",
        "rota_alternativa": "NAO"
    },
    {
        "categoria": "PASSAGEM_OBSTRUIDA",
        "impacto": "ALTO",
        "rota_alternativa": "SIM"
    },
    {
        "categoria": "RAMPA_INADEQUADA",
        "impacto": "ALTO",
        "rota_alternativa": "NAO"
    },
    {
        "categoria": "TRAVESSIA_INADEQUADA",
        "impacto": "MEDIO",
        "rota_alternativa": "SIM"
    }
])

previsoes = modelo_prioridade.predict(novas_ocorrencias)

for i, prioridade in enumerate(previsoes):
    print(f"Ocorrência {i + 1}:")
    print(f"Categoria: {novas_ocorrencias.iloc[i]['categoria']}")
    print(f"Impacto: {novas_ocorrencias.iloc[i]['impacto']}")
    print(f"Rota alternativa: {novas_ocorrencias.iloc[i]['rota_alternativa']}")
    print(f"Prioridade prevista: {prioridade}\n")

Ocorrência 1:
Categoria: CALCADA_DANIFICADA
Impacto: BAIXO
Rota alternativa: SIM
Prioridade prevista: BAIXA

Ocorrência 2:
Categoria: RAMPA_AUSENTE
Impacto: MEDIO
Rota alternativa: NAO
Prioridade prevista: ALTA

Ocorrência 3:
Categoria: PASSAGEM_OBSTRUIDA
Impacto: ALTO
Rota alternativa: SIM
Prioridade prevista: ALTA

Ocorrência 4:
Categoria: RAMPA_INADEQUADA
Impacto: ALTO
Rota alternativa: NAO
Prioridade prevista: ALTA

Ocorrência 5:
Categoria: TRAVESSIA_INADEQUADA
Impacto: MEDIO
Rota alternativa: SIM
Prioridade prevista: MEDIA



## 7. Exportação do Modelo

Após o treinamento e os testes, o modelo é exportado utilizando a biblioteca Joblib.

O arquivo gerado é `modelo_prioridade.joblib`, armazenado na pasta `api`.

A exportação permite que o modelo seja carregado posteriormente pela API Flask para classificar a prioridade de novas ocorrências, sem a necessidade de repetir o treinamento.

In [8]:
joblib.dump(
    modelo_prioridade,
    "../api/modelo_prioridade.joblib"
)

print("Modelo salvo com sucesso!")

Modelo salvo com sucesso!


## 8. Conclusão

O desenvolvimento deste modelo permitiu criar um classificador capaz de determinar o nível de prioridade de ocorrências relacionadas à acessibilidade urbana.

A classificação utiliza as informações de categoria, impacto e existência de rota alternativa, processadas por meio de um OneHotEncoder e classificadas com o algoritmo DecisionTreeClassifier.

Nos testes realizados com o conjunto de dados utilizado, o modelo obteve acurácia de 1,00 (100%).

Entretanto, como o dataset utilizado possui dados sintéticos, esse resultado não garante o mesmo desempenho em situações reais. A utilização em cenários reais exige validação com dados representativos de ocorrências urbanas.